# Kebutuhan sistem

In [2]:
import pandas as pd
from sqlalchemy import create_engine, text
from urllib.parse import quote_plus

# Lingkungan pengembangan

In [3]:
user = "root"
password = quote_plus("")      # isi password MySQL di dalam tanda kutip kalau ada
host = "localhost"
port = 3306

In [5]:
db_src = "classicmodels"      # database operasional (sumber)
db_dw  = "classicmodels_dw"   # database warehouse (tujuan)

# Engine

In [6]:
eng_src = create_engine(f"mysql+pymysql://{user}:{password}@{host}:{port}/{db_src}")
eng_dw  = create_engine(f"mysql+pymysql://{user}:{password}@{host}:{port}/{db_dw}")
print("Engine siap")

Engine siap


In [7]:
tes_src = pd.read_sql("SELECT COUNT(*) AS jumlah FROM customers", eng_src)
print(f"Sumber OK — jumlah customers: {tes_src['jumlah'][0]}")

Sumber OK — jumlah customers: 122


In [8]:
tes_dw = pd.read_sql("SHOW TABLES", eng_dw)
print(f"DW OK — jumlah tabel saat ini: {len(tes_dw)}")

DW OK — jumlah tabel saat ini: 0


In [9]:
versi = pd.read_sql("SELECT VERSION() AS versi", eng_src)
print(f"MySQL versi: {versi['versi'][0]}")

MySQL versi: 8.0.30


# Pembuatan tabel dimensi dan fakta

In [16]:
# Drop dulu kalau ada sisa percobaan sebelumnya, biar bersih
tabel_dw = [
    'fakta_pembayaran',
    'fakta_penjualan',
    'dim_waktu',
    'dim_produk',
    'dim_pelanggan',
    'dim_karyawan'
]

with eng_dw.begin() as conn:
    conn.execute(text("SET FOREIGN_KEY_CHECKS = 0"))
    for t in tabel_dw:
        conn.execute(text(f"DROP TABLE IF EXISTS {t}"))
    conn.execute(text("SET FOREIGN_KEY_CHECKS = 1"))

print("Tabel lama (kalau ada) sudah dibersihkan")

Tabel lama (kalau ada) sudah dibersihkan


In [17]:
ddl_dimensi = """
CREATE TABLE dim_karyawan (
    id              INT PRIMARY KEY,
    employee_number INT,
    nama            VARCHAR(100),
    jabatan         VARCHAR(50),
    email           VARCHAR(100),
    kota_kantor     VARCHAR(50),
    negara_kantor   VARCHAR(50)
);

CREATE TABLE dim_pelanggan (
    id                   INT PRIMARY KEY,
    customer_number      INT,
    nama                 VARCHAR(50),
    kontak_nama_depan    VARCHAR(50),
    kontak_nama_belakang VARCHAR(50),
    telepon              VARCHAR(50),
    alamat               VARCHAR(50),
    kota                 VARCHAR(50),
    negara               VARCHAR(50),
    batas_kredit         DECIMAL(10,2),
    karyawan_id          INT,
    FOREIGN KEY (karyawan_id) REFERENCES dim_karyawan(id)
);

CREATE TABLE dim_produk (
    id           INT PRIMARY KEY,
    product_code VARCHAR(15),
    nama_produk  VARCHAR(70),
    vendor       VARCHAR(50),
    lini_produk  VARCHAR(50),
    harga_beli   DECIMAL(10,2),
    msrp         DECIMAL(10,2)
);

CREATE TABLE dim_waktu (
    id         INT PRIMARY KEY,
    tanggal    DATE,
    tahun      INT,
    bulan      INT,
    kuartal    INT,
    nama_hari  VARCHAR(10)
);
"""

with eng_dw.begin() as conn:
    for stmt in ddl_dimensi.strip().split(";"):
        if stmt.strip():
            conn.execute(text(stmt))

print("4 tabel dimensi berhasil dibuat")

4 tabel dimensi berhasil dibuat


In [18]:
ddl_fakta = """
CREATE TABLE fakta_penjualan (
    id               INT PRIMARY KEY AUTO_INCREMENT,
    pelanggan_id     INT,
    produk_id        INT,
    karyawan_id      INT,
    waktu_id         INT,
    nomor_pesanan    INT,
    jumlah           INT,
    harga_satuan     DECIMAL(10,2),
    total_penjualan  DECIMAL(10,2),
    FOREIGN KEY (pelanggan_id) REFERENCES dim_pelanggan(id),
    FOREIGN KEY (produk_id)    REFERENCES dim_produk(id),
    FOREIGN KEY (karyawan_id)  REFERENCES dim_karyawan(id),
    FOREIGN KEY (waktu_id)     REFERENCES dim_waktu(id)
);

CREATE TABLE fakta_pembayaran (
    id            INT PRIMARY KEY AUTO_INCREMENT,
    pelanggan_id  INT,
    waktu_id      INT,
    nomor_cek     VARCHAR(50),
    jumlah_bayar  DECIMAL(10,2),
    FOREIGN KEY (pelanggan_id) REFERENCES dim_pelanggan(id),
    FOREIGN KEY (waktu_id)     REFERENCES dim_waktu(id)
);
"""

with eng_dw.begin() as conn:
    for stmt in ddl_fakta.strip().split(";"):
        if stmt.strip():
            conn.execute(text(stmt))

print("2 tabel fakta berhasil dibuat")

2 tabel fakta berhasil dibuat


In [19]:
with eng_dw.connect() as conn:
    hasil = pd.read_sql("SHOW TABLES", conn)
print(f"Total tabel di DW: {len(hasil)}")
print(hasil)

Total tabel di DW: 6
  Tables_in_classicmodels_dw
0               dim_karyawan
1              dim_pelanggan
2                 dim_produk
3                  dim_waktu
4           fakta_pembayaran
5            fakta_penjualan


In [20]:
pd.read_sql("DESCRIBE fakta_penjualan", eng_dw)

,Field,Type,Null,Key,Default,Extra
0,id,int,NO,PRI,None,auto_increment
1,pelanggan_id,int,YES,MUL,None,
2,produk_id,int,YES,MUL,None,
3,karyawan_id,int,YES,MUL,None,
4,waktu_id,int,YES,MUL,None,
5,nomor_pesanan,int,YES,,None,
6,jumlah,int,YES,,None,
7,harga_satuan,"decimal(10,2)",YES,,None,
8,total_penjualan,"decimal(10,2)",YES,,None,


# ETL dimensi karyawan

In [21]:
df_karyawan = pd.read_sql("""
    SELECT
        ROW_NUMBER() OVER (ORDER BY e.employeeNumber) AS id,
        e.employeeNumber                                AS employee_number,
        CONCAT(e.firstName, ' ', e.lastName)            AS nama,
        e.jobTitle                                      AS jabatan,
        e.email                                         AS email,
        o.city                                          AS kota_kantor,
        o.country                                       AS negara_kantor
    FROM employees e
    JOIN offices o ON e.officeCode = o.officeCode
""", eng_src)

df_karyawan

,id,employee_number,nama,jabatan,email,kota_kantor,negara_kantor
0,1,1002,Diane Murphy,President,dmurphy@classicmodelcars.com,San Francisco,USA
1,2,1056,Mary Patterson,VP Sales,mpatterso@classicmodelcars.com,San Francisco,USA
2,3,1076,Jeff Firrelli,VP Marketing,jfirrelli@classicmodelcars.com,San Francisco,USA
3,4,1088,William Patterson,Sales Manager (APAC),wpatterson@classicmodelcars.com,Sydney,Australia
4,5,1102,Gerard Bondur,Sale Manager (EMEA),gbondur@classicmodelcars.com,Paris,France
5,6,1143,Anthony Bow,Sales Manager (NA),abow@classicmodelcars.com,San Francisco,USA
6,7,1165,Leslie Jennings,Sales Rep,ljennings@classicmodelcars.com,San Francisco,USA
7,8,1166,Leslie Thompson,Sales Rep,lthompson@classicmodelcars.com,San Francisco,USA
8,9,1188,Julie Firrelli,Sales Rep,jfirrelli@classicmodelcars.com,Boston,USA
9,10,1216,Steve Patterson,Sales Rep,spatterson@classicmodelcars.com,Boston,USA


In [22]:
with eng_dw.begin() as conn:
    conn.execute(text("DELETE FROM dim_karyawan"))

df_karyawan.to_sql('dim_karyawan', con=eng_dw, if_exists='append', index=False)
print(f"{len(df_karyawan)} baris dimuat ke dim_karyawan")

23 baris dimuat ke dim_karyawan


In [23]:
pd.read_sql("SELECT * FROM dim_karyawan LIMIT 5", eng_dw)

,id,employee_number,nama,jabatan,email,kota_kantor,negara_kantor
0,1,1002,Diane Murphy,President,dmurphy@classicmodelcars.com,San Francisco,USA
1,2,1056,Mary Patterson,VP Sales,mpatterso@classicmodelcars.com,San Francisco,USA
2,3,1076,Jeff Firrelli,VP Marketing,jfirrelli@classicmodelcars.com,San Francisco,USA
3,4,1088,William Patterson,Sales Manager (APAC),wpatterson@classicmodelcars.com,Sydney,Australia
4,5,1102,Gerard Bondur,Sale Manager (EMEA),gbondur@classicmodelcars.com,Paris,France


# ETL dimensi pelanggan

In [25]:
df_pelanggan = pd.read_sql(f"""
    SELECT
        ROW_NUMBER() OVER (ORDER BY c.customerNumber) AS id,
        c.customerNumber                              AS customer_number,
        c.customerName                                AS nama,
        c.contactFirstName                            AS kontak_nama_depan,
        c.contactLastName                             AS kontak_nama_belakang,
        c.phone                                       AS telepon,
        c.addressLine1                                AS alamat,
        c.city                                        AS kota,
        c.country                                     AS negara,
        c.creditLimit                                 AS batas_kredit,
        k.id                                          AS karyawan_id
    FROM {db_src}.customers c
    LEFT JOIN {db_dw}.dim_karyawan k ON k.employee_number = c.salesRepEmployeeNumber
""", eng_dw)

df_pelanggan

,id,customer_number,nama,kontak_nama_depan,kontak_nama_belakang,telepon,alamat,kota,negara,batas_kredit,karyawan_id
0,1,103,Atelier graphique,Carine,Schmitt,40.32.2555,"54, rue Royale",Nantes,France,21000.0,14.0
1,2,112,Signal Gift Stores,Jean,King,7025551838,8489 Strong St.,Las Vegas,USA,71800.0,8.0
2,3,114,"Australian Collectors, Co.",Peter,Ferguson,03 9520 4555,636 St Kilda Road,Melbourne,Australia,117300.0,18.0
3,4,119,La Rochelle Gifts,Janine,Labrune,40.67.8555,"67, rue des Cinquante Otages",Nantes,France,118200.0,14.0
4,5,121,Baane Mini Imports,Jonas,Bergulfsen,07-98 9555,Erling Skakkes gate 78,Stavern,Norway,81700.0,17.0
...,...,...,...,...,...,...,...,...,...,...,...
117,118,486,Motor Mint Distributors Inc.,Rosa,Salazar,2155559857,11328 Douglas Av.,Philadelphia,USA,72600.0,12.0
118,119,487,Signal Collectibles Ltd.,Sue,Taylor,4155554312,2793 Furth Circle,Brisbane,USA,60300.0,7.0
119,120,489,"Double Decker Gift Stores, Ltd",Thomas,Smith,(171) 555-7555,120 Hanover Sq.,London,UK,43300.0,16.0
120,121,495,Diecast Collectables,Valarie,Franco,6175552555,6251 Ingle Ln.,Boston,USA,85100.0,9.0


In [26]:
print(f"Total pelanggan: {len(df_pelanggan)}")
print(f"Pelanggan tanpa sales rep (karyawan_id NULL): {df_pelanggan['karyawan_id'].isna().sum()}")
print()
print("Contoh yang NULL:")
print(df_pelanggan[df_pelanggan['karyawan_id'].isna()][['nama', 'kota', 'negara']].head())

Total pelanggan: 122
Pelanggan tanpa sales rep (karyawan_id NULL): 22

Contoh yang NULL:
                          nama       kota     negara
6           Havel & Zbyszek Co   Warszawa     Poland
21           Porto Imports Co.     Lisboa   Portugal
36  Asian Shopping Network, Co  Singapore  Singapore
41             Natürlich Autos  Cunewalde    Germany
44               ANG Resellers     Madrid      Spain


In [28]:
with eng_dw.begin() as conn:
    conn.execute(text("DELETE FROM dim_pelanggan"))

df_pelanggan.to_sql('dim_pelanggan', con=eng_dw, if_exists='append', index=False)
print(f"{len(df_pelanggan)} baris dimuat ke dim_pelanggan")

122 baris dimuat ke dim_pelanggan


In [29]:
pd.read_sql("SELECT * FROM dim_pelanggan LIMIT 5", eng_dw)

,id,customer_number,nama,kontak_nama_depan,kontak_nama_belakang,telepon,alamat,kota,negara,batas_kredit,karyawan_id
0,1,103,Atelier graphique,Carine,Schmitt,40.32.2555,"54, rue Royale",Nantes,France,21000.0,14
1,2,112,Signal Gift Stores,Jean,King,7025551838,8489 Strong St.,Las Vegas,USA,71800.0,8
2,3,114,"Australian Collectors, Co.",Peter,Ferguson,03 9520 4555,636 St Kilda Road,Melbourne,Australia,117300.0,18
3,4,119,La Rochelle Gifts,Janine,Labrune,40.67.8555,"67, rue des Cinquante Otages",Nantes,France,118200.0,14
4,5,121,Baane Mini Imports,Jonas,Bergulfsen,07-98 9555,Erling Skakkes gate 78,Stavern,Norway,81700.0,17


# ETL dimensi produk

In [30]:
df_produk = pd.read_sql("""
    SELECT
        ROW_NUMBER() OVER (ORDER BY productCode) AS id,
        productCode                              AS product_code,
        productName                              AS nama_produk,
        productVendor                            AS vendor,
        productLine                              AS lini_produk,
        buyPrice                                 AS harga_beli,
        MSRP                                     AS msrp
    FROM products
""", eng_src)

df_produk

,id,product_code,nama_produk,vendor,lini_produk,harga_beli,msrp
0,1,S10_1678,1969 Harley Davidson Ultimate Chopper,Min Lin Diecast,Motorcycles,48.81,95.70
1,2,S10_1949,1952 Alpine Renault 1300,Classic Metal Creations,Classic Cars,98.58,214.30
2,3,S10_2016,1996 Moto Guzzi 1100i,Highway 66 Mini Classics,Motorcycles,68.99,118.94
3,4,S10_4698,2003 Harley-Davidson Eagle Drag Bike,Red Start Diecast,Motorcycles,91.02,193.66
4,5,S10_4757,1972 Alfa Romeo GTA,Motor City Art Classics,Classic Cars,85.68,136.00
...,...,...,...,...,...,...,...
105,106,S700_3505,The Titanic,Carousel DieCast Legends,Ships,51.09,100.17
106,107,S700_3962,The Queen Mary,Welly Diecast Productions,Ships,53.63,99.31
107,108,S700_4002,American Airlines: MD-11S,Second Gear Diecast,Planes,36.27,74.03
108,109,S72_1253,Boeing X-32A JSF,Motor City Art Classics,Planes,32.77,49.66


In [32]:
with eng_dw.begin() as conn:
    conn.execute(text("DELETE FROM dim_produk"))

df_produk.to_sql('dim_produk', con=eng_dw, if_exists='append', index=False)
print(f"{len(df_produk)} baris dimuat ke dim_produk")

110 baris dimuat ke dim_produk


In [33]:
pd.read_sql("SELECT * FROM dim_produk LIMIT 5", eng_dw)

,id,product_code,nama_produk,vendor,lini_produk,harga_beli,msrp
0,1,S10_1678,1969 Harley Davidson Ultimate Chopper,Min Lin Diecast,Motorcycles,48.81,95.70
1,2,S10_1949,1952 Alpine Renault 1300,Classic Metal Creations,Classic Cars,98.58,214.30
2,3,S10_2016,1996 Moto Guzzi 1100i,Highway 66 Mini Classics,Motorcycles,68.99,118.94
3,4,S10_4698,2003 Harley-Davidson Eagle Drag Bike,Red Start Diecast,Motorcycles,91.02,193.66
4,5,S10_4757,1972 Alfa Romeo GTA,Motor City Art Classics,Classic Cars,85.68,136.00


In [34]:
rentang = pd.read_sql("""
    SELECT MIN(d) AS min_tgl, MAX(d) AS max_tgl FROM (
        SELECT orderDate AS d FROM orders
        UNION
        SELECT paymentDate FROM payments
    ) t
""", eng_src)

print(rentang)

      min_tgl     max_tgl
0  2003-01-06  2005-06-09


In [35]:
min_tgl = pd.to_datetime(rentang['min_tgl'][0])
max_tgl = pd.to_datetime(rentang['max_tgl'][0])

semua_tanggal = pd.date_range(min_tgl, max_tgl, freq='D')

df_waktu = pd.DataFrame({
    'id':        range(1, len(semua_tanggal) + 1),
    'tanggal':   semua_tanggal.date,     # ← ambil .date biar jadi DATE (bukan DATETIME)
    'tahun':     semua_tanggal.year,
    'bulan':     semua_tanggal.month,
    'kuartal':   semua_tanggal.quarter,
    'nama_hari': semua_tanggal.day_name()
})

print(f"{len(df_waktu)} tanggal di-generate dari {min_tgl.date()} s/d {max_tgl.date()}")
df_waktu.head()

886 tanggal di-generate dari 2003-01-06 s/d 2005-06-09


,id,tanggal,tahun,bulan,kuartal,nama_hari
0,1,2003-01-06,2003,1,1,Monday
1,2,2003-01-07,2003,1,1,Tuesday
2,3,2003-01-08,2003,1,1,Wednesday
3,4,2003-01-09,2003,1,1,Thursday
4,5,2003-01-10,2003,1,1,Friday


In [36]:
with eng_dw.begin() as conn:
    conn.execute(text("DELETE FROM dim_waktu"))

df_waktu.to_sql('dim_waktu', con=eng_dw, if_exists='append', index=False)
print(f"{len(df_waktu)} baris dimuat ke dim_waktu")

886 baris dimuat ke dim_waktu


In [37]:
pd.read_sql("SELECT * FROM dim_waktu LIMIT 5", eng_dw)

,id,tanggal,tahun,bulan,kuartal,nama_hari
0,1,2003-01-06,2003,1,1,Monday
1,2,2003-01-07,2003,1,1,Tuesday
2,3,2003-01-08,2003,1,1,Wednesday
3,4,2003-01-09,2003,1,1,Thursday
4,5,2003-01-10,2003,1,1,Friday


In [38]:
for tabel in ['dim_karyawan', 'dim_pelanggan', 'dim_produk', 'dim_waktu']:
    jml = pd.read_sql(f"SELECT COUNT(*) AS n FROM {tabel}", eng_dw)['n'][0]
    print(f"{tabel:15s} → {jml:>4} baris")

dim_karyawan    →   23 baris
dim_pelanggan   →  122 baris
dim_produk      →  110 baris
dim_waktu       →  886 baris


In [39]:
# Cek jumlah di DataFrame vs jumlah di tabel
for nama_df, nama_tabel in [
    ('df_karyawan',  'dim_karyawan'),
    ('df_pelanggan', 'dim_pelanggan'),
    ('df_produk',    'dim_produk'),
    ('df_waktu',     'dim_waktu'),
]:
    df = globals()[nama_df]
    jml_df = len(df)
    jml_tbl = pd.read_sql(f"SELECT COUNT(*) AS n FROM {nama_tabel}", eng_dw)['n'][0]
    status = "✅" if jml_df == jml_tbl else "⚠️ BEDA"
    print(f"{status} {nama_tabel:15s} → DataFrame: {jml_df:>4} | Tabel: {jml_tbl:>4}")

✅ dim_karyawan    → DataFrame:   23 | Tabel:   23
✅ dim_pelanggan   → DataFrame:  122 | Tabel:  122
✅ dim_produk      → DataFrame:  110 | Tabel:  110
✅ dim_waktu       → DataFrame:  886 | Tabel:  886


# ETL fakta penjualan

In [40]:
df_penjualan_raw = pd.read_sql("""
    SELECT
        o.customerNumber                            AS customer_number,
        od.productCode                              AS product_code,
        c.salesRepEmployeeNumber                    AS employee_number,
        o.orderDate                                 AS tanggal,
        o.orderNumber                               AS nomor_pesanan,
        od.quantityOrdered                          AS jumlah,
        od.priceEach                                AS harga_satuan,
        od.quantityOrdered * od.priceEach           AS total_penjualan
    FROM orderdetails od
    JOIN orders o    ON od.orderNumber = o.orderNumber
    JOIN customers c ON o.customerNumber = c.customerNumber
    WHERE o.status <> 'Cancelled'
""", eng_src)

print(f"Total baris penjualan (exclude Cancelled): {len(df_penjualan_raw)}")
df_penjualan_raw.head()

Total baris penjualan (exclude Cancelled): 2917


,customer_number,product_code,employee_number,tanggal,nomor_pesanan,jumlah,harga_satuan,total_penjualan
0,363,S18_1749,1216,2003-01-06,10100,30,136.00,4080.00
1,363,S18_2248,1216,2003-01-06,10100,50,55.09,2754.50
2,363,S18_4409,1216,2003-01-06,10100,22,75.46,1660.12
3,363,S24_3969,1216,2003-01-06,10100,49,35.29,1729.21
4,128,S18_2325,1504,2003-01-09,10101,25,108.06,2701.50


In [41]:
# Mapping pelanggan: customer_number → id
map_pelanggan = df_pelanggan[['customer_number', 'id']].rename(columns={'id': 'pelanggan_id'})

# Mapping produk: product_code → id
map_produk = df_produk[['product_code', 'id']].rename(columns={'id': 'produk_id'})

# Mapping karyawan: employee_number → id
map_karyawan = df_karyawan[['employee_number', 'id']].rename(columns={'id': 'karyawan_id'})

# Mapping waktu: tanggal → id
map_waktu = df_waktu[['tanggal', 'id']].rename(columns={'id': 'waktu_id'})
# Pastikan tipe tanggal konsisten
map_waktu['tanggal'] = pd.to_datetime(map_waktu['tanggal'])

print("Mapping surrogate key siap")
print(f"  Pelanggan: {len(map_pelanggan)} baris")
print(f"  Produk   : {len(map_produk)} baris")
print(f"  Karyawan : {len(map_karyawan)} baris")
print(f"  Waktu    : {len(map_waktu)} baris")

Mapping surrogate key siap
  Pelanggan: 122 baris
  Produk   : 110 baris
  Karyawan : 23 baris
  Waktu    : 886 baris


In [42]:
# Pastikan tipe tanggal di df_penjualan_raw juga datetime
df_penjualan_raw['tanggal'] = pd.to_datetime(df_penjualan_raw['tanggal'])

df_penjualan = (df_penjualan_raw
    .merge(map_pelanggan, left_on='customer_number', right_on='customer_number', how='left')
    .merge(map_produk,    left_on='product_code',    right_on='product_code',    how='left')
    .merge(map_karyawan,  left_on='employee_number', right_on='employee_number', how='left')
    .merge(map_waktu,     left_on='tanggal',         right_on='tanggal',         how='left')
)

print(f"Total baris setelah merge: {len(df_penjualan)}")
print(f"pelanggan_id NULL: {df_penjualan['pelanggan_id'].isna().sum()}")
print(f"produk_id NULL   : {df_penjualan['produk_id'].isna().sum()}")
print(f"karyawan_id NULL : {df_penjualan['karyawan_id'].isna().sum()}")
print(f"waktu_id NULL    : {df_penjualan['waktu_id'].isna().sum()}")

Total baris setelah merge: 2917
pelanggan_id NULL: 0
produk_id NULL   : 0
karyawan_id NULL : 0
waktu_id NULL    : 0


In [43]:
# Cek kalau ada NULL yang tidak seharusnya
for col in ['pelanggan_id', 'produk_id', 'waktu_id']:
    n_null = df_penjualan[col].isna().sum()
    if n_null > 0:
        print(f"⚠️ {col} NULL: {n_null} baris — perlu dicek!")
        # Tampilkan contoh
        print(df_penjualan[df_penjualan[col].isna()].head())
    else:
        print(f"✅ {col}: tidak ada NULL")

print(f"\nℹ️ karyawan_id NULL: {df_penjualan['karyawan_id'].isna().sum()} baris (normal — customer tanpa sales rep)")

✅ pelanggan_id: tidak ada NULL
✅ produk_id: tidak ada NULL
✅ waktu_id: tidak ada NULL

ℹ️ karyawan_id NULL: 0 baris (normal — customer tanpa sales rep)


In [45]:
# Pilih kolom sesuai struktur tabel fakta_penjualan
kolom_final = ['pelanggan_id', 'produk_id', 'karyawan_id', 'waktu_id',
                'nomor_pesanan', 'jumlah', 'harga_satuan', 'total_penjualan']

df_penjualan_final = df_penjualan[kolom_final].copy()

# Konversi ke int (biar tidak ada float karena merge)
for col in ['pelanggan_id', 'produk_id', 'karyawan_id', 'waktu_id',
            'nomor_pesanan', 'jumlah']:
    df_penjualan_final[col] = df_penjualan_final[col].astype('Int64')  # Int64 = nullable int

# Load ke DW
with eng_dw.begin() as conn:
    conn.execute(text("DELETE FROM fakta_penjualan"))

df_penjualan_final.to_sql('fakta_penjualan', con=eng_dw, if_exists='append', index=False)
print(f"{len(df_penjualan_final)} baris dimuat ke fakta_penjualan")

2917 baris dimuat ke fakta_penjualan


In [46]:
# Cek jumlah baris
jml_df  = len(df_penjualan_final)
jml_tbl = pd.read_sql("SELECT COUNT(*) AS n FROM fakta_penjualan", eng_dw)['n'][0]
print(f"DataFrame: {jml_df} | Tabel: {jml_tbl} | {'✅ cocok' if jml_df == jml_tbl else '⚠️ BEDA'}")

# Cek total penjualan (validasi)
total_dw = pd.read_sql("SELECT SUM(total_penjualan) AS t FROM fakta_penjualan", eng_dw)['t'][0]
total_src = df_penjualan_raw['total_penjualan'].sum()
print(f"\nTotal penjualan di DW     : {total_dw:,.2f}")
print(f"Total penjualan di sumber : {total_src:,.2f}")
print(f"Selisih                   : {abs(total_dw - total_src):,.2f}")

DataFrame: 2917 | Tabel: 2917 | ✅ cocok

Total penjualan di DW     : 9,365,336.43
Total penjualan di sumber : 9,365,336.43
Selisih                   : 0.00


In [47]:
pd.read_sql("SELECT * FROM fakta_penjualan LIMIT 5", eng_dw)

,id,pelanggan_id,produk_id,karyawan_id,waktu_id,nomor_pesanan,jumlah,harga_satuan,total_penjualan
0,2918,86,23,10,1,10100,30,136.00,4080.00
1,2919,86,27,10,1,10100,50,55.09,2754.50
2,2920,86,50,10,1,10100,22,75.46,1660.12
3,2921,86,80,10,1,10100,49,35.29,1729.21
4,2922,8,29,17,4,10101,25,108.06,2701.50


#  ETL fakta pembayaran

In [48]:
df_pembayaran_raw = pd.read_sql("""
    SELECT
        p.customerNumber  AS customer_number,
        p.paymentDate     AS tanggal,
        p.checkNumber     AS nomor_cek,
        p.amount          AS jumlah_bayar
    FROM payments p
""", eng_src)

print(f"Total baris pembayaran: {len(df_pembayaran_raw)}")
df_pembayaran_raw.head()

Total baris pembayaran: 273


,customer_number,tanggal,nomor_cek,jumlah_bayar
0,103,2004-10-19,HQ336336,6066.78
1,103,2003-06-05,JM555205,14571.44
2,103,2004-12-18,OM314933,1676.14
3,112,2004-12-17,BO864823,14191.12
4,112,2003-06-06,HQ55022,32641.98


In [49]:
# Pastikan tipe tanggal datetime
df_pembayaran_raw['tanggal'] = pd.to_datetime(df_pembayaran_raw['tanggal'])

df_pembayaran = (df_pembayaran_raw
    .merge(map_pelanggan, left_on='customer_number', right_on='customer_number', how='left')
    .merge(map_waktu,     left_on='tanggal',         right_on='tanggal',         how='left')
)

print(f"Total baris setelah merge: {len(df_pembayaran)}")
print(f"pelanggan_id NULL: {df_pembayaran['pelanggan_id'].isna().sum()}")
print(f"waktu_id NULL    : {df_pembayaran['waktu_id'].isna().sum()}")

Total baris setelah merge: 273
pelanggan_id NULL: 0
waktu_id NULL    : 0


In [50]:
kolom_final_bayar = ['pelanggan_id', 'waktu_id', 'nomor_cek', 'jumlah_bayar']
df_pembayaran_final = df_pembayaran[kolom_final_bayar].copy()

# Konversi FK ke nullable int
df_pembayaran_final['pelanggan_id'] = df_pembayaran_final['pelanggan_id'].astype('Int64')
df_pembayaran_final['waktu_id']     = df_pembayaran_final['waktu_id'].astype('Int64')

# Load
with eng_dw.begin() as conn:
    conn.execute(text("DELETE FROM fakta_pembayaran"))

df_pembayaran_final.to_sql('fakta_pembayaran', con=eng_dw, if_exists='append', index=False)
print(f"{len(df_pembayaran_final)} baris dimuat ke fakta_pembayaran")

273 baris dimuat ke fakta_pembayaran


In [51]:
jml_df  = len(df_pembayaran_final)
jml_tbl = pd.read_sql("SELECT COUNT(*) AS n FROM fakta_pembayaran", eng_dw)['n'][0]
print(f"DataFrame: {jml_df} | Tabel: {jml_tbl} | {'✅ cocok' if jml_df == jml_tbl else '⚠️ BEDA'}")

total_dw  = pd.read_sql("SELECT SUM(jumlah_bayar) AS t FROM fakta_pembayaran", eng_dw)['t'][0]
total_src = df_pembayaran_raw['jumlah_bayar'].sum()
print(f"\nTotal pembayaran di DW     : {total_dw:,.2f}")
print(f"Total pembayaran di sumber : {total_src:,.2f}")
print(f"Selisih                    : {abs(total_dw - total_src):,.2f}")

DataFrame: 273 | Tabel: 273 | ✅ cocok

Total pembayaran di DW     : 8,853,839.23
Total pembayaran di sumber : 8,853,839.23
Selisih                    : 0.00


In [52]:
pd.read_sql("SELECT * FROM fakta_pembayaran LIMIT 5", eng_dw)

,id,pelanggan_id,waktu_id,nomor_cek,jumlah_bayar
0,1,1,653,HQ336336,6066.78
1,2,1,151,JM555205,14571.44
2,3,1,713,OM314933,1676.14
3,4,2,712,BO864823,14191.12
4,5,2,152,HQ55022,32641.98


In [ ]:
print("=" * 50)
print("RINGKASAN AKHIR DATA WAREHOUSE")
print("=" * 50)

for tabel in ['dim_karyawan', 'dim_pelanggan', 'dim_produk', 'dim_waktu',
                'fakta_penjualan', 'fakta_pembayaran']:
    jml = pd.read_sql(f"SELECT COUNT(*) AS n FROM {tabel}", eng_dw)['n'][0]
    tipe = "Dimensi" if tabel.startswith('dim_') else "Fakta"
    print(f"  [{tipe:7s}] {tabel:20s} → {jml:>5} baris")

RINGKASAN AKHIR DATA WAREHOUSE
  [Dimensi] dim_karyawan         →    23 baris
  [Dimensi] dim_pelanggan        →   122 baris
  [Dimensi] dim_produk           →   110 baris
  [Dimensi] dim_waktu            →   886 baris
  [Fakta  ] fakta_penjualan      →  2917 baris
  [Fakta  ] fakta_pembayaran     →   273 baris


#  5 Pertanyaan Bisnis

In [ ]:
# soal 1
q1 = pd.read_sql("""
    SELECT SUM(total_penjualan) AS total_penjualan
    FROM fakta_penjualan
""", eng_dw)

print("=" * 50)
print("Q1: Total Nilai Penjualan di Classic Models")
print("=" * 50)
print(f"Total penjualan: ${q1['total_penjualan'][0]:,.2f}")
q1

Q1: Total Nilai Penjualan di Classic Models
Total penjualan: $9,365,336.43


,total_penjualan
0,9365336.43


In [55]:
# soal 2
q2 = pd.read_sql("""
    SELECT
        p.nama_produk,
        p.vendor,
        SUM(f.total_penjualan) AS total_transaksi
    FROM fakta_penjualan f
    JOIN dim_produk p ON f.produk_id = p.id
    GROUP BY p.nama_produk, p.vendor
    ORDER BY total_transaksi DESC
    LIMIT 1
""", eng_dw)

print("=" * 50)
print("Q2: Produk & Vendor dengan Nilai Transaksi Terbesar")
print("=" * 50)
print(f"Produk : {q2['nama_produk'][0]}")
print(f"Vendor : {q2['vendor'][0]}")
print(f"Total  : ${q2['total_transaksi'][0]:,.2f}")
q2

Q2: Produk & Vendor dengan Nilai Transaksi Terbesar
Produk : 1992 Ferrari 360 Spider red
Vendor : Unimax Art Galleries
Total  : $271,014.78


,nama_produk,vendor,total_transaksi
0,1992 Ferrari 360 Spider red,Unimax Art Galleries,271014.78


In [56]:
# soal 3
q3 = pd.read_sql("""
    WITH total_penjualan AS (
        SELECT pelanggan_id, SUM(total_penjualan) AS total
        FROM fakta_penjualan
        GROUP BY pelanggan_id
    ),
    total_pembayaran AS (
        SELECT pelanggan_id, SUM(jumlah_bayar) AS total
        FROM fakta_pembayaran
        GROUP BY pelanggan_id
    )
    SELECT
        p.nama,
        p.kota,
        p.negara,
        COALESCE(j.total, 0) AS total_penjualan,
        COALESCE(b.total, 0) AS total_pembayaran,
        COALESCE(j.total, 0) - COALESCE(b.total, 0) AS hutang
    FROM dim_pelanggan p
    LEFT JOIN total_penjualan j  ON p.id = j.pelanggan_id
    LEFT JOIN total_pembayaran b ON p.id = b.pelanggan_id
    WHERE COALESCE(j.total, 0) - COALESCE(b.total, 0) > 0
    ORDER BY hutang DESC
""", eng_dw)

print("=" * 50)
print("Q3: Pelanggan yang Masih Memiliki Hutang")
print("=" * 50)
print(f"Jumlah pelanggan berhutang: {len(q3)}")
print(f"Total hutang keseluruhan  : ${q3['hutang'].sum():,.2f}")
print("\nTop 10 pelanggan dengan hutang terbesar:")
q3.head(10)

Q3: Pelanggan yang Masih Memiliki Hutang
Jumlah pelanggan berhutang: 16
Total hutang keseluruhan  : $511,497.20

Top 10 pelanggan dengan hutang terbesar:


,nama,kota,negara,total_penjualan,total_pembayaran,hutang
0,The Sharp Gifts Warehouse,San Jose,USA,143536.27,59551.38,83984.89
1,Euro+ Shopping Channel,Madrid,Spain,773624.18,715738.98,57885.20
2,Salzburg Collectables,Salzburg,Austria,137480.07,85060.00,52420.07
3,Gifts4AllAges.com,Boston,USA,84340.32,33533.47,50806.85
4,Tekni Collectables Inc.,Newark,USA,81806.55,38281.51,43525.04
5,Souveniers And Things Co.,Chatswood,Australia,133907.12,91655.61,42251.51
6,La Rochelle Gifts,Nantes,France,158573.12,116949.68,41623.44
7,L'ordine Souveniers,Reggio Emilia,Italy,125505.57,90143.31,35362.26
8,"Extreme Desk Decorations, Ltd",Wellington,New Zealand,90332.38,66704.94,23627.44
9,"Volvo Model Replicas, Co",Luleå,Sweden,66694.82,43680.65,23014.17


In [57]:
# soal 4
q4 = pd.read_sql("""
    SELECT
        k.nama,
        k.jabatan,
        k.kota_kantor,
        SUM(f.total_penjualan) AS total_order
    FROM fakta_penjualan f
    JOIN dim_karyawan k ON f.karyawan_id = k.id
    GROUP BY k.id, k.nama, k.jabatan, k.kota_kantor
    ORDER BY total_order DESC
    LIMIT 1
""", eng_dw)

print("=" * 50)
print("Q4: Karyawan dengan Nilai Order Terbesar")
print("=" * 50)
print(f"Nama    : {q4['nama'][0]}")
print(f"Jabatan : {q4['jabatan'][0]}")
print(f"Kantor  : {q4['kota_kantor'][0]}")
print(f"Total   : ${q4['total_order'][0]:,.2f}")
q4

Q4: Karyawan dengan Nilai Order Terbesar
Nama    : Gerard Hernandez
Jabatan : Sales Rep
Kantor  : Paris
Total   : $1,211,512.45


,nama,jabatan,kota_kantor,total_order
0,Gerard Hernandez,Sales Rep,Paris,1211512.45


In [58]:
# soal 5
q5 = pd.read_sql("""
    WITH karyawan_terbaik AS (
        SELECT f.karyawan_id
        FROM fakta_penjualan f
        WHERE f.karyawan_id IS NOT NULL
        GROUP BY f.karyawan_id
        ORDER BY SUM(f.total_penjualan) DESC
        LIMIT 1
    )
    SELECT
        p.nama           AS nama_customer,
        p.kota,
        p.negara,
        k.nama           AS nama_karyawan,
        SUM(f.total_penjualan) AS total_order
    FROM fakta_penjualan f
    JOIN dim_pelanggan p ON f.pelanggan_id = p.id
    JOIN dim_karyawan  k ON f.karyawan_id = k.id
    WHERE f.karyawan_id = (SELECT karyawan_id FROM karyawan_terbaik)
    GROUP BY p.id, p.nama, p.kota, p.negara, k.nama
    ORDER BY total_order DESC
    LIMIT 1
""", eng_dw)

print("=" * 50)
print("Q5: Customer dengan Order Terbesar dari Karyawan Terbaik")
print("=" * 50)
print(f"Karyawan terbaik : {q5['nama_karyawan'][0]}")
print(f"Customer         : {q5['nama_customer'][0]}")
print(f"Lokasi           : {q5['kota'][0]}, {q5['negara'][0]}")
print(f"Total order      : ${q5['total_order'][0]:,.2f}")
q5

Q5: Customer dengan Order Terbesar dari Karyawan Terbaik
Karyawan terbaik : Gerard Hernandez
Customer         : Euro+ Shopping Channel
Lokasi           : Madrid, Spain
Total order      : $773,624.18


,nama_customer,kota,negara,nama_karyawan,total_order
0,Euro+ Shopping Channel,Madrid,Spain,Gerard Hernandez,773624.18


In [59]:
print("=" * 60)
print("RINGKASAN JAWABAN 5 PERTANYAAN BISNIS")
print("=" * 60)
print(f"\nQ1. Total penjualan       : ${q1['total_penjualan'][0]:,.2f}")
print(f"\nQ2. Produk terlaris       : {q2['nama_produk'][0]}")
print(f"    Vendor                : {q2['vendor'][0]}")
print(f"    Total transaksi       : ${q2['total_transaksi'][0]:,.2f}")
print(f"\nQ3. Pelanggan berhutang   : {len(q3)} pelanggan")
print(f"    Total hutang          : ${q3['hutang'].sum():,.2f}")
print(f"    Hutang terbesar       : {q3['nama'][0]} (${q3['hutang'][0]:,.2f})")
print(f"\nQ4. Karyawan terbaik      : {q4['nama'][0]}")
print(f"    Total order           : ${q4['total_order'][0]:,.2f}")
print(f"\nQ5. Customer top dari {q5['nama_karyawan'][0]}:")
print(f"    {q5['nama_customer'][0]} — ${q5['total_order'][0]:,.2f}")
print("\n" + "=" * 60)

RINGKASAN JAWABAN 5 PERTANYAAN BISNIS

Q1. Total penjualan       : $9,365,336.43

Q2. Produk terlaris       : 1992 Ferrari 360 Spider red
    Vendor                : Unimax Art Galleries
    Total transaksi       : $271,014.78

Q3. Pelanggan berhutang   : 16 pelanggan
    Total hutang          : $511,497.20
    Hutang terbesar       : The Sharp Gifts Warehouse ($83,984.89)

Q4. Karyawan terbaik      : Gerard Hernandez
    Total order           : $1,211,512.45

Q5. Customer top dari Gerard Hernandez:
    Euro+ Shopping Channel — $773,624.18

